In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax
from torch_geometric.nn import HeteroConv, GATConv, HGTConv
from torch_geometric.nn import GCNConv, GATConv, GINConv

/opt/anaconda3/envs/firegnn/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Message level gating

In [ ]:
class ConditionalMessageGate(nn.Module):
    """
    Edge-wise message gate with different logic for:
    - non-patient -> non-patient
    - non-patient -> patient
    - patient -> non-patient (optional)
    """
    def __init__(self):
        super().__init__()

        # For protein → patient
        self.theta_np_p = nn.Parameter(torch.zeros(1))
        self.alpha_np_p = nn.Parameter(torch.ones(1))

        # For non-patient → non-patient 
        self.theta_np_np = nn.Parameter(torch.zeros(1))
        self.alpha_np_np = nn.Parameter(torch.ones(1))

    def forward(
        self,
        src_is_patient: torch.BoolTensor,   # [E]
        dst_is_patient: torch.BoolTensor,   # [E]
        src_relevance: torch.Tensor,        # [E] (undefined for patient, masked)
        dst_label: torch.Tensor,            # [E] (only valid if dst is patient)
        edge_weight: torch.Tensor           # [E]
    ):
        """
        Returns:
            gate: [E] in (0,1)
        """
        gate = torch.ones_like(edge_weight)

        # -------- non-patient -> patient --------
        mask_np_p = (~src_is_patient) & dst_is_patient
        if mask_np_p.any():
            # Only allow flow to AD patients (y=1)
            score = src_relevance[mask_np_p] * edge_weight[mask_np_p]
            cond = score * dst_label[mask_np_p]   # dst_label ∈ {0,1}
            gate[mask_np_p] = torch.sigmoid(
                self.alpha_np_p * (cond - self.theta_np_p)
            )

        # -------- non-patient -> non-patient --------
        mask_np_np = (~src_is_patient) & (~dst_is_patient)
        if mask_np_np.any():
            score = src_relevance[mask_np_np] * edge_weight[mask_np_np]
            gate[mask_np_np] = torch.sigmoid(
                self.alpha_np_np * (score - self.theta_np_np)
            )

        # -------- patient -> non-patient --------
        # Optional: dampen or leave as 1.0
        # mask_p_np = src_is_patient & (~dst_is_patient)
        # gate[mask_p_np] = 0.5

        return gate


In [ ]:
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax

class ConditionalGATMessageLayer(MessagePassing):
    def __init__(self, in_channels, out_channels, heads=1, dropout=0.0):
        super().__init__(aggr='add', node_dim=0)

        self.heads = heads
        self.out_channels = out_channels

        self.lin = nn.Linear(in_channels, heads * out_channels, bias=False)
        self.att = nn.Parameter(torch.Tensor(1, heads, 2 * out_channels))

        self.gate = ConditionalMessageGate()
        self.dropout = dropout
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.lin.weight)
        nn.init.xavier_uniform_(self.att)

    def forward(
        self,
        x,
        edge_index,
        edge_weight,
        src_is_patient,
        dst_is_patient,
        src_relevance,
        dst_label
    ):
        H = self.lin(x).view(-1, self.heads, self.out_channels)
        return self.propagate(
            edge_index,
            x=H,
            edge_weight=edge_weight,
            src_is_patient=src_is_patient,
            dst_is_patient=dst_is_patient,
            src_relevance=src_relevance,
            dst_label=dst_label
        )

    def message(
        self,
        x_i,
        x_j,
        edge_weight,
        src_is_patient,
        dst_is_patient,
        src_relevance,
        dst_label,
        index
    ):
        # ---- attention (standard GAT) ----
        cat = torch.cat([x_i, x_j], dim=-1)
        e = (cat * self.att).sum(dim=-1)
        e = F.leaky_relu(e, 0.2)

        alpha = softmax(e, index)
        alpha = F.dropout(alpha, p=self.dropout, training=self.training)

        # ---- conditional gate ----
        g = self.gate(
            src_is_patient=src_is_patient,
            dst_is_patient=dst_is_patient,
            src_relevance=src_relevance,
            dst_label=dst_label,
            edge_weight=edge_weight
        )

        # ---- gated message ----
        return x_j * alpha.unsqueeze(-1) * g.unsqueeze(-1)


In [ ]:
from torch_geometric.nn import HeteroConv

class ConditionalHeteroGAT_MessageGate(nn.Module):
    def __init__(self, data, in_channels, hidden_channels, out_channels, heads=2):
        super().__init__()

        self.convs = nn.ModuleList()
        self.convs.append(HeteroConv({
            edge_type: ConditionalGATMessageLayer(
                in_channels,
                hidden_channels,
                heads=heads
            )
            for edge_type in data.edge_types
        }, aggr='sum'))

        self.convs.append(HeteroConv({
            edge_type: ConditionalGATMessageLayer(
                hidden_channels * heads,
                out_channels,
                heads=1
            )
            for edge_type in data.edge_types
        }, aggr='sum'))

    def forward(
        self,
        x_dict,
        edge_index_dict,
        edge_weight_dict,
        relevance_dict,
        y_dict,
        is_patient_dict
    ):
        for conv in self.convs:
            x_dict = conv(
                x_dict,
                edge_index_dict,
                edge_weight_dict,
                is_patient_dict,
                relevance_dict,
                y_dict
            )
        return x_dict


### Conditional Attention coefficient term

In [ ]:
class ConditionalGATAttentionLayer(MessagePassing):
    def __init__(self, in_channels, out_channels, heads=1, dropout=0.0):
        super().__init__(node_dim=0, aggr='add')
        self.heads = heads
        self.out_channels = out_channels

        self.lin = nn.Linear(in_channels, heads * out_channels, bias=False)
        self.att = nn.Parameter(torch.Tensor(1, heads, 2 * out_channels))

        self.theta = nn.Parameter(torch.zeros(1))
        self.alpha = nn.Parameter(torch.ones(1))

        self.dropout = dropout
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.lin.weight)
        nn.init.xavier_uniform_(self.att)

    def forward(self, x, edge_index, src_relevance, edge_weight):
        H = self.lin(x).view(-1, self.heads, self.out_channels)
        return self.propagate(edge_index, x=H, src_relevance=src_relevance, edge_weight=edge_weight)

    def message(self, x_i, x_j, src_relevance, edge_weight, index):
        # Standard GAT attention
        cat = torch.cat([x_i, x_j], dim=-1)
        e = (cat * self.att).sum(dim=-1)
        e = F.leaky_relu(e, 0.2)

        # Disease-aware gate
        gate = torch.sigmoid(self.alpha * (src_relevance * edge_weight - self.theta))
        e = e + torch.log(gate.unsqueeze(-1) + 1e-6)

        alpha = softmax(e, index)
        alpha = F.dropout(alpha, p=self.dropout, training=self.training)

        return x_j * alpha.unsqueeze(-1)

class ConditionalHeteroGAT_AttentionGate(nn.Module):
    def __init__(self, data, in_channels, hidden_channels, out_channels, heads=2):
        super().__init__()
        self.convs = nn.ModuleList()

        self.convs.append(HeteroConv({
            edge_type: ConditionalGATAttentionLayer(
                in_channels,
                hidden_channels,
                heads=heads
            )
            for edge_type in data.edge_types
        }, aggr='sum'))

        self.convs.append(HeteroConv({
            edge_type: ConditionalGATAttentionLayer(
                hidden_channels * heads,
                out_channels,
                heads=1
            )
            for edge_type in data.edge_types
        }, aggr='sum'))

    def forward(self, x_dict, edge_index_dict, edge_weight_dict, relevance_dict):
        for conv in self.convs:
            x_dict = conv(
                x_dict,
                edge_index_dict,
                relevance_dict,
                edge_weight_dict
            )
        return x_dict

